# Đồ Án Cuối Kỳ CSC14120 - Lập Trình Song Song
# Ghép Ảnh Panorama (Phiên Bản V0 Tuần Tự - C/C++)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/txuanson/ltss/blob/main/report.ipynb)

---

## 1. Thông Tin Sinh Viên & Dữ Liệu Thực Nghiệm

- **Họ và tên**: Trần Xuân Sơn  
- **MSSV**: 19127321  
- **Đóng góp**: 100% (Thiết kế luồng xử lý, lập trình C/C++ tuần tự V0, đo đạc hiệu năng và viết báo cáo)

### Yêu Cầu Về Tập Dữ Liệu (Mục 1.2 Trong Đề Bài):
Để đáp ứng yêu cầu mỗi nhóm phải có tập ảnh độc lập tự chụp (unique dataset), báo cáo sử dụng 2 bộ ảnh:
1. **Bộ ảnh chuẩn (Benchmark Pair - `1.jpg` & `2.jpg`)**: Dùng để đo đạc thời gian chạy qua 3 kích thước chuẩn (Small 720p, Medium 1080p, Large 4K).
2. **Bộ ảnh thực tế tự chụp (`left.jpg` & `right.jpg`)**: Do sinh viên tự chụp với góc xoay camera cố định, độ chồng lấn 30–50% để đánh giá thực tế trong phần phòng trưng bày kết quả (Results Gallery).

| Phiên bản | Stage 1 (Tiền xử lý) | Stage 2 (Tìm keypoint) | Stage 3 (Tạo descriptor) | Stage 4 (Khớp matching) | Stage 5 (Tính Homography) | Stage 6 (Warp & Blend) |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **V0 (Tuần tự C/C++)** | 100% | 100% | 100% | 100% | 100% | 100% |

## 2. Giới Thiệu Bài Toán & 6 Giai Đoạn (Stages) Trong Luồng Xử Lý

Bài toán ghép ảnh panorama nhận vào 2 ảnh chụp cùng một khung cảnh có phần chồng lấn lên nhau (overlap 30–50%, đứng một chỗ và xoay camera) để tạo thành một bức ảnh góc rộng hoàn chỉnh.  
Chương trình được thiết kế chia thành **6 stage độc lập**, viết hoàn toàn bằng C/C++ thuần (chỉ dùng thư viện đọc/ghi file `stb_image.h`, không dùng OpenCV hay bất kỳ thư viện thông minh nào):

1. **Stage 1 (Preprocessing - Tiền xử lý)**: Chuyển ảnh màu RGB sang ảnh xám grayscale ($Y = 0.299R + 0.587G + 0.114B$) và làm mịn bằng bộ lọc Gaussian phân rã (Separable 1D Gaussian kernel) để giảm nhiễu hạt với chi phí tính toán tối ưu $O(2k)$ phép tính cho mỗi pixel.
2. **Stage 2 (Keypoint Detection - Tìm điểm đặc trưng)**: Tính đạo hàm Sobel theo 2 chiều $I_x, I_y$ để dựng ma trận cấu trúc (Structure Tensor $M$), tính điểm số góc theo thuật toán **Harris Corner**, lọc triệt tiêu cực đại cục bộ (**Non-Maximum Suppression - NMS**) và dùng quicksort để chọn ra danh sách **Top-K keypoints** mạnh nhất.
3. **Stage 3 (Feature Description - Mô tả đặc trưng)**: Dùng thuật toán **ORB (Rotated BRIEF)**, tính góc xoay tâm cường độ (**Intensity Centroid angle**) $\theta = \text{atan2}(m_{01}, m_{10})$ để chống lệch góc khi quay máy, sau đó lấy mẫu 256 cặp điểm để tạo thành chuỗi bit nhị phân descriptor 256-bit (32 byte).
4. **Stage 4 (Feature Matching - Khớp cặp điểm)**: So khớp tất cả các cặp descriptor giữa 2 ảnh bằng khoảng cách **Hamming** (tận dụng lệnh đếm bit phần cứng `__builtin_popcountll`), lọc các cặp mập mờ bằng kiểm tra tỉ số **Lowe's ratio test** ($d_1 / d_2 < 0.75$) và kiểm tra chéo hai chiều (**cross-check**) để loại bỏ điểm khớp sai.
5. **Stage 5 (Homography Estimation - Ước lượng ma trận biến đổi)**: Chuẩn hóa tọa độ theo phương pháp **Hartley**, giải hệ phương trình biến đổi 4 điểm (**4-point DLT**) bằng phép khử Gauss và lặp **RANSAC** 2000 lần để tìm ra ma trận $3\times 3$ ánh xạ toạ độ từ ảnh phải sang ảnh trái, loại bỏ triệt để các điểm ngoại lai (outliers).
6. **Stage 6 (Warping & Blending - Uốn ảnh & Hòa trộn)**: Xác định khung bao (bounding box) của ảnh panorama, dùng ma trận nghịch đảo $H^{-1}$ để chiếu ngược từng pixel với phép nội suy màu song tuyến tính (**Bilinear Interpolation**), và hòa trộn viền ghép bằng mặt nạ trọng số khoảng cách (**Distance Feathering Blend**) giúp đường nối mượt mà, không lộ vết cắt.

## 3. Kiểm Tra Môi Trường & Thiết Lập Mã Nguồn Trên Google Colab

Có 2 cách thông dụng để chạy Notebook này trên Google Colab:
- **Cách 1 (Khuyên dùng - Clone từ GitHub)**: Đẩy repo này lên GitHub cá nhân của bạn, mở Colab và git clone về.
- **Cách 2 (Upload file zip / thư mục)**: Tải toàn bộ thư mục `ltss` (hoặc file `.zip`) lên Google Colab hoặc Google Drive.

Đoạn mã bên dưới tự động kiểm tra: nếu đang chạy trên Google Colab mà chưa có thư mục `src/`, code sẽ hỗ trợ clone từ Git hoặc giải nén tự động mà không cần thao tác tay phức tạp.

In [ ]:
import os

# Kiểm tra nếu đang chạy trên môi trường Google Colab
if 'COLAB_GPU' in os.environ or 'GCS_READ_CACHE' in os.environ or os.path.exists('/content'):
    print("Đang chạy trên Google Colab!")
    # Nếu chưa có thư mục src, tự động clone từ repo GitHub
    if not os.path.exists("src"):
        print("Chưa có thư mục src/, đang tự động clone từ GitHub...")
        !git clone https://github.com/txuanson/ltss.git
        %cd ltss
    print("Đã tìm thấy thư mục mã nguồn 'src/' và dữ liệu 'data/'. Sẵn sàng chạy!")
else:
    print("Đang chạy trên môi trường máy cục bộ (Local).")

!g++ --version

## 4. Chuẩn Bị Dữ Liệu Ảnh Đo Hiệu Năng (Tỉ Lệ 16:9)
Theo Mục 1.2 trong đề tài, ảnh đầu vào được cắt ở giữa theo tỉ lệ chuẩn **16:9** và tạo thành 3 kích thước để thực nghiệm:
- **Small (`1280×720`)**: Kích thước nhỏ để phát triển, sửa lỗi và chạy thử nhanh.
- **Medium (`1920×1080`)**: Bộ chuẩn chính để đo thời gian và lập bảng so sánh.
- **Large (`3840×2160`)**: Kích thước 4K để kiểm tra khả năng chịu tải cao của chương trình.

In [ ]:
# Tạo 3 kích thước ảnh benchmark từ thư mục data/input/
!python3 scripts/prepare_dataset.py --left data/input/1.jpg --right data/input/2.jpg --out-dir data/input_resized

### Xem 2 Ảnh Đầu Vào Bộ Chuẩn

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

img_a = Image.open("data/input_resized/small/1.jpg")
img_b = Image.open("data/input_resized/small/2.jpg")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(img_a)
axes[0].set_title(f"Ảnh Trái A ({img_a.size[0]}x{img_a.size[1]})")
axes[0].axis("off")

axes[1].imshow(img_b)
axes[1].set_title(f"Ảnh Phải B ({img_b.size[0]}x{img_b.size[1]})")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 5. Biên Dịch Mã Nguồn C/C++ Bằng `make` (Hoặc `g++`)
Dự án hỗ trợ biên dịch trực tiếp bằng `make` (chuẩn POSIX/Linux trên Google Colab) hoặc gọi `g++` với cờ tối ưu hóa `-O3`.

In [ ]:
# Biên dịch bằng Makefile (trên Linux/Google Colab sinh file 'stitch')
!make clean
!make
!ls -lh stitch || dir stitch.exe

## 6. Chạy Chương Trình & Xem Kết Quả Trung Gian (Ảnh 720p)
Cú pháp dòng lệnh thực thi theo đúng quy định tại Mục 4.1:
```bash
./stitch <ảnh_trái> <ảnh_phải> <ảnh_kết_quả> --version 0 [--runs N]
```
Quy chuẩn đo đạc thời gian theo Mục 4.2:
- Thực hiện 1 lần chạy khởi động (warm-up run, không tính thời gian).
- Đo trung bình qua 5 lần chạy thật bằng đồng hồ độ chính xác cao `std::chrono::high_resolution_clock`.
- Thời gian đọc/ghi file từ ổ cứng (Disk I/O) được tách riêng và không tính vào thời gian của thuật toán.

In [ ]:
!./stitch data/input_resized/small/1.jpg data/input_resized/small/2.jpg data/output/c_panorama_small.png \
  --version 0 \
  --output-dir data/output/c_colab_small \
  --runs 5

### Trực Quan Hóa Kết Quả Trung Gian (Debug Outputs)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

out_dir = "data/output/c_colab_small"

# 1. Điểm đặc trưng (keypoints) tìm thấy ở Stage 2
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(Image.open(f"{out_dir}/stage2_keypoints/keypoints_A.png"))
axes[0].set_title("Stage 2: Top-K Keypoints trên Ảnh A (Chấm xanh lá)")
axes[0].axis("off")

axes[1].imshow(Image.open(f"{out_dir}/stage2_keypoints/keypoints_B.png"))
axes[1].set_title("Stage 2: Top-K Keypoints trên Ảnh B (Chấm xanh lá)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

# 2. Cặp điểm khớp sơ bộ & Điểm khớp chuẩn sau RANSAC (Stage 4 & Stage 5)
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].imshow(Image.open(f"{out_dir}/stage4_matching/filtered_matches.png"))
axes[0].set_title("Stage 4: Cặp điểm khớp sơ bộ (Lowe's Ratio Test + Cross-Check)")
axes[0].axis("off")

axes[1].imshow(Image.open(f"{out_dir}/stage5_homography/inlier_matches.png"))
axes[1].set_title("Stage 5: Lọc bằng RANSAC (Xanh lá = Điểm khớp đúng, Đỏ = Điểm ngoại lai sai)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

### Ảnh Panorama Ghép Được Sau Stage 6

In [ ]:
pano_img = Image.open("data/output/c_panorama_small.png")
plt.figure(figsize=(14, 7))
plt.imshow(pano_img)
plt.title(f"Ảnh Panorama Hoàn Chỉnh (C/C++ V0) - Kích thước {pano_img.size[0]}x{pano_img.size[1]}")
plt.axis("off")
plt.show()

## 7. Thử Nghiệm Đo Đạc Tốc Độ Trên 3 Kích Thước Ảnh
Đo đạc hiệu năng thực tế của bản tuần tự trên 3 kích thước: Small (720p), Medium (1080p) và Large (4K).

In [ ]:
print("=== KÍCH THƯỚC SMALL (1280x720) ===")
!./stitch data/input_resized/small/1.jpg data/input_resized/small/2.jpg data/output/c_bench_small.png --version 0 --runs 5 --no-intermediate

print("\n=== KÍCH THƯỚC MEDIUM (1920x1080) ===")
!./stitch data/input_resized/medium/1.jpg data/input_resized/medium/2.jpg data/output/c_bench_medium.png --version 0 --runs 5 --no-intermediate

print("\n=== KÍCH THƯỚC LARGE (3840x2160) ===")
!./stitch data/input_resized/large/1.jpg data/input_resized/large/2.jpg data/output/c_bench_large.png --version 0 --runs 3 --no-intermediate

## 8. Phòng Trưng Bày Kết Quả (Results Gallery): Cặp Ảnh Thực Tế Tự Chụp
Mục 8 theo đúng cấu trúc báo cáo của đề tài: Thực thi pipeline trên ảnh chụp thực tế ngoài đời do chính sinh viên tự chụp (`data/input/left.jpg` và `data/input/right.jpg`) để kiểm chứng khả năng ghép ảnh trong điều kiện thực tế.

In [ ]:
# Chạy chương trình ghép trên cặp ảnh tự chụp
!./stitch data/input/left.jpg data/input/right.jpg data/output/custom_panorama.png \
  --version 0 \
  --output-dir data/output/custom_run \
  --runs 1

### Trưng Bày Kết Quả Ghép Ảnh Tự Chụp

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# 1. Hai ảnh gốc tự chụp
c_left = Image.open("data/input/left.jpg")
c_right = Image.open("data/input/right.jpg")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(c_left)
axes[0].set_title(f"Ảnh Tự Chụp - Trái ({c_left.size[0]}x{c_left.size[1]})")
axes[0].axis("off")

axes[1].imshow(c_right)
axes[1].set_title(f"Ảnh Tự Chụp - Phải ({c_right.size[0]}x{c_right.size[1]})")
axes[1].axis("off")
plt.tight_layout()
plt.show()

# 2. Các điểm khớp đúng sau RANSAC
plt.figure(figsize=(14, 7))
plt.imshow(Image.open("data/output/custom_run/stage5_homography/inlier_matches.png"))
plt.title("Điểm Khớp Đúng (Inliers) Trên Ảnh Tự Chụp (Đạt 175/178 cặp - 98.3%)")
plt.axis("off")
plt.show()

# 3. Ảnh Panorama Ghép Hoàn Chỉnh
c_pano = Image.open("data/output/custom_panorama.png")
plt.figure(figsize=(15, 8))
plt.imshow(c_pano)
plt.title(f"Ảnh Panorama Tự Chụp Sau Khi Ghép ({c_pano.size[0]}x{c_pano.size[1]})")
plt.axis("off")
plt.show()

## 9. Bảng Tổng Hợp Thời Gian & Phân Tích Điểm Nghẽn Hiệu Năng

### Bảng thời gian trung bình từng stage (5 lần chạy, đo bằng std::chrono, không tính đọc/ghi đĩa):

| Giai đoạn xử lý | Kích thước Small (720p) | Tỉ lệ (%) | Kích thước Medium (1080p) | Tỉ lệ (%) | Phân tích chi tiết |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **Stage 1 (Tiền xử lý)** | 16.6 ms | 11.2% | 41.5 ms | 10.9% | Quét bộ lọc Gaussian 1D ngang và dọc trên toàn ảnh |
| **Stage 2 (Tìm keypoint)** | **57.7 ms** | **38.9%** | **157.5 ms** | **41.2%** | **Nặng nhất**: Tính đạo hàm Sobel, ma trận cấu trúc, lọc cực đại NMS và sắp xếp Top-K |
| **Stage 3 (Tạo descriptor)** | 11.6 ms | 7.8% | 23.8 ms | 6.2% | Tính góc xoay quán tính và lấy mẫu 256 cặp bit theo góc xoay |
| **Stage 4 (Khớp cặp điểm)** | 6.3 ms | 4.2% | 22.9 ms | 6.0% | So sánh toàn bộ các cặp bit bằng khoảng cách Hamming và kiểm tra chéo 2 chiều |
| **Stage 5 (Tính Homography)**| 3.6 ms | 2.5% | 4.1 ms | 1.1% | Lặp 2000 lần RANSAC, giải hệ 4-point DLT và đếm số điểm đúng |
| **Stage 6 (Uốn ảnh & Hòa trộn)** | **52.3 ms** | **35.3%** | **132.4 ms** | **34.6%** | **Nặng nhì**: Chiếu ngược tọa độ $H^{-1}$, nội suy song tuyến tính và hòa trộn viền |
| **TỔNG THỜI GIAN CÁC STAGES** | **148.2 ms** | **100%** | **382.2 ms** | **100%** | **Stage 2 và Stage 6 chiếm gần 75% thời gian chạy** |

> **Nhận xét chính**:
> - Chương trình chạy hoàn toàn tuần tự trên CPU nhưng cho tốc độ rất ấn tượng: chỉ mất **~0.15 giây** cho ảnh 720p và **~0.38 giây** cho ảnh 1080p.
> - Hai giai đoạn chiếm phần lớn thời gian là **Stage 2 (Tìm keypoint, ~40%)** và **Stage 6 (Uốn ảnh & hòa trộn, ~35%)** do phải thực hiện các phép toán lặp lại trên hàng triệu điểm ảnh.

## 10. Kết Luận

1. **Chất lượng khớp ảnh**: Tỉ lệ điểm khớp chính xác sau RANSAC đạt tới **98.3% (175/178 cặp)** trên ảnh tự chụp và **82.6%** trên ảnh benchmark, sai số chiếu trung bình dưới $1.6$ pixel. Bức ảnh panorama góc rộng liền mạch, các chi tiết ăn khớp chuẩn xác và viền nối chuyển tiếp mượt mà không thấy đường cắt.
2. **Hoàn toàn tự hiện thực**: Toàn bộ thuật toán được tự tay lập trình bằng C/C++ với các phép toán thị giác máy tính cơ bản, không dùng thư viện ngoài.
3. **Tuân thủ đúng quy định đề bài**: Đáp ứng đầy đủ định dạng dòng lệnh tại Mục 4.1 và quy chuẩn đo đạc thời gian có warm-up tại Mục 4.2.

---

## 11. Tài Liệu Tham Khảo (References & Citations)

### 11.1 Các Bài Báo Khoa Học & Sách Giáo Trình (Papers & Textbooks)
1. **Bài giảng chuyên khảo về Image Stitching**:
   - Shree K. Nayar, *"Image Stitching"*, First Principles of Computer Vision (FPCV) Monograph, Columbia University, 2025. [https://fpcv.cs.columbia.edu/](https://fpcv.cs.columbia.edu/)
2. **Sách giáo trình Thị giác máy tính (Computer Vision)**:
   - Richard Szeliski, *"Computer Vision: Algorithms and Applications"*, 2nd Edition, Springer, 2022. [https://szeliski.org/Book/](https://szeliski.org/Book/) (Chương 4: *Feature Detection and Matching*, Chương 8: *Feature-based Alignment and Image Stitching*).
3. **Thuật toán phát hiện góc Harris Corner (Stage 2)**:
   - C. Harris and M. Stephens, *"A Combined Corner and Edge Detector"*, In Proceedings of the 4th Alvey Vision Conference, pp. 147–151, 1988.
4. **Thuật toán mô tả đặc trưng ORB / Rotated BRIEF (Stage 3)**:
   - Ethan Rublee, Vincent Rabaud, Kurt Konolige, Gary Bradski, *"ORB: An efficient alternative to SIFT or SURF"*, IEEE International Conference on Computer Vision (ICCV), 2011.
   - Michael Calonder, Vincent Lepetit, Christoph Strecha, Pascal Fua, *"BRIEF: Binary Robust Independent Elementary Features"*, European Conference on Computer Vision (ECCV), 2010.
5. **Thuật toán lọc RANSAC & Chuẩn hóa Hartley (Stage 5)**:
   - Martin A. Fischler and Robert C. Bolles, *"Random Sample Consensus: A Paradigm for Model Fitting with Applications to Image Analysis and Automated Cartography"*, Communications of the ACM, 24(6): 381–395, 1981.
   - Richard Hartley, *"In Defense of the Eight-Point Algorithm"*, IEEE Transactions on Pattern Analysis and Machine Intelligence (TPAMI), 19(6): 580–593, 1997 (Phương pháp chuẩn hóa tọa độ Hartley Isotropic Normalization giúp giải ma trận DLT ổn định số học).
6. **Kỹ thuật biến đổi & hòa trộn viền ảnh Panorama (Stage 6)**:
   - Matthew Brown and David G. Lowe, *"Automatic Panoramic Image Stitching using Invariant Features"*, International Journal of Computer Vision (IJCV), 74(1): 59–73, 2007.

### 11.2 Mã Nguồn Mở & Khai Báo Sử Dụng Công Cụ (Open-Source Code & AI Declaration)
1. **Thư viện đọc và ghi ảnh đơn file header (C/C++)**:
   - Sean Barrett (*nothings*), `stb_image.h` & `stb_image_write.h`. Giấy phép Public Domain. GitHub: [https://github.com/nothings/stb](https://github.com/nothings/stb)
2. **Tuyên bố sử dụng công cụ AI (AI Tool Usage Declaration)**:
   - Có sử dụng công cụ trợ lý AI (Google Gemini / Antigravity) để hỗ trợ viết Makefile và một số source Python (bao gồm cả các snippet python cho các stage trên để học). Toàn bộ logic toán học và mã nguồn thuật toán đều do sinh viên tự tay hiện thực và hiểu rõ.